# 14 - Advanced Evaluation & Model Comparison

The final, comprehensive evaluation. Loads **every trained model** (the 8 core + the
extensions, when present) and runs the full battery, re-scoring frozen models only
(no re-training):

- **A. Comparison leaderboard** (from `all_metrics.json`) — full table + headline charts +
  rating-vs-ranking trade-off + F1@K curves.
- **B. NDCG@K & AUC** — ranking metrics that are less brittle than F1@K.
- **C. Segmented RMSE** — by user activity & item popularity (where does each model win?).
- **D. Beyond-accuracy** — catalogue coverage, intra-list diversity, novelty.
- **E. Bootstrap confidence intervals** — is the Stacked/Dual-Head gap real?
- **F. Cold-start simulation** — content models given only 3 ratings of history.
- **G. Full-catalogue sanity** — a no-sampling ranking pass for a few models.

All deep sections run on **bounded samples** (config constants at the top of each); raise
them for tighter estimates.

In [1]:
import sys
sys.path.insert(0, "../src")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from hybrid_recsys.pipeline.data import load_processed
from hybrid_recsys.pipeline.splits import load_splits
from hybrid_recsys.evaluation.report import full_metrics, save_metric, top_n
from hybrid_recsys.config import RANDOM_STATE

FIGURES_DIR = Path("../artifacts/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.write_html(str(FIGURES_DIR / f"{name}.html"))
    try:
        fig.write_image(str(FIGURES_DIR / f"{name}.png"), width=1100, height=550, scale=2)
    except Exception:
        pass
    fig.show()


EVAL_USERS, N_NEGATIVES = 1_000, 100
rng = np.random.default_rng(RANDOM_STATE)

movies           = load_processed("movies")
train, val, test = load_splits()
train_val        = pd.concat([train, val], ignore_index=True)
all_movie_ids    = movies["movieId"].values
user_ratings_map = (
    train.groupby("userId").apply(lambda d: dict(zip(d["movieId"], d["rating"]))).to_dict()
)
eval_user_ids = rng.choice(
    test["userId"].unique(), size=min(EVAL_USERS, test["userId"].nunique()), replace=False
)
test_sample = test[test["userId"].isin(eval_user_ids)]
demo_user   = max(eval_user_ids, key=lambda u: len(user_ratings_map.get(u, {})))
print(f"Loaded splits - train {len(train):,}, test {len(test):,}; demo user = {int(demo_user)}")


Loaded splits - train 19,936,012, test 2,633,022; demo user = 127311


In [2]:
def ranking_curve(metrics, title):
    rows = [{"K": k, "Metric": m.capitalize(), "Value": metrics[f"k{k}"][m]}
            for k in [5, 10, 20] for m in ["precision", "recall", "f1"]]
    return px.line(pd.DataFrame(rows), x="K", y="Value", color="Metric", markers=True,
                   title=f"Ranking metrics @ K - {title}")


def error_hist(preds, title):
    err = test["rating"].to_numpy() - preds
    err = err[~np.isnan(err)]
    fig = px.histogram(err, nbins=50, title=f"Rating error (true - predicted) - {title}")
    fig.update_layout(showlegend=False, xaxis_title="true - predicted", bargap=0.02)
    return fig


def show_example(predict_fn, n=10, n_candidates=3000):
    seen = set(user_ratings_map.get(demo_user, {}))
    cand = rng.choice(all_movie_ids, size=min(n_candidates, len(all_movie_ids)), replace=False)
    recs = top_n(predict_fn, demo_user, seen, cand, movies, n=n)
    hist = (
        pd.DataFrame({"movieId": list(seen),
                      "rating": [user_ratings_map[demo_user][m] for m in seen]})
        .merge(movies[["movieId", "clean_title", "genres"]], on="movieId", how="left")
        .sort_values("rating", ascending=False).head(n)
    )
    return hist, recs


def star_graph(center, leaves, weights, title, name):
    k = len(leaves)
    angles = np.linspace(0, 2 * np.pi, k, endpoint=False)
    lx, ly = np.cos(angles), np.sin(angles)
    edge_x, edge_y = [], []
    for x, y in zip(lx, ly):
        edge_x += [0, x, None]
        edge_y += [0, y, None]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                             line=dict(color="#cccccc", width=1), hoverinfo="none"))
    fig.add_trace(go.Scatter(
        x=lx, y=ly, mode="markers+text",
        marker=dict(size=16, color=list(weights), colorscale="Teal",
                    showscale=True, colorbar=dict(title="sim")),
        text=[f"{l}<br>{w:.2f}" for l, w in zip(leaves, weights)],
        textposition="top center", hoverinfo="text"))
    fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers+text",
                             marker=dict(size=26, color="#EF553B"),
                             text=[center], textposition="bottom center", hoverinfo="text"))
    fig.update_layout(title=title, showlegend=False,
                      xaxis=dict(visible=False), yaxis=dict(visible=False))
    save_fig(fig, name)
    return fig


## Load all models & build scoring functions

In [3]:
from hybrid_recsys.models.content import ContentBasedRecommender
from hybrid_recsys.models.collaborative import SVDModel, ItemKNNModel, UserKNNModel
from hybrid_recsys.models.hybrid import WeightedHybrid, StackedHybrid, DualHeadHybrid
from hybrid_recsys.models.lightgcn import LightGCNRecommender
from hybrid_recsys.config import ARTIFACTS_MODELS

urm = user_ratings_map
ip = train.groupby("movieId").size().to_dict()
uc = train.groupby("userId").size().to_dict()
mp = max(ip.values()); gm = float(train["rating"].mean())

cb = ContentBasedRecommender.load(); uk = UserKNNModel.load(); ik = ItemKNNModel.load()
svd = SVDModel.load(); wh = WeightedHybrid.load(); sh = StackedHybrid.load()

def stacked_pred(u, i):
    base = np.array([cb.predict(urm.get(u, {}), i), uk.predict(u, i), ik.predict(u, i), svd.predict(u, i)], float)
    return sh.predict_one(u, i, base)

# RATE_FN: rating-prediction score (RMSE-based analyses). RANK_FN: ranking score.
RATE_FN = {
    "Global Mean":     lambda u, i: gm,
    "Content-Based":   lambda u, i: cb.predict(urm.get(u, {}), i),
    "User-Based k-NN": lambda u, i: uk.predict(u, i),
    "Item-Based k-NN": lambda u, i: ik.predict(u, i),
    "SVD":             lambda u, i: svd.predict(u, i),
    "Weighted Hybrid": lambda u, i: wh.predict(u, i, urm.get(u, {})),
    "Stacked Hybrid":  stacked_pred,
}
RANK_FN = dict(RATE_FN)
RANK_FN["Popularity"] = lambda u, i: 0.5 + 4.5 * (ip.get(i, 0) / mp)

genome_ok = lg_ok = False
try:
    cb_g = ContentBasedRecommender.load(path=ARTIFACTS_MODELS / "content_genome_model.joblib")
    RATE_FN["Content-Based (Genome)"] = lambda u, i: cb_g.predict(urm.get(u, {}), i)
    RANK_FN["Content-Based (Genome)"] = RATE_FN["Content-Based (Genome)"]; genome_ok = True
except Exception as e:
    print("genome content not loaded:", e)
try:
    cb_e = ContentBasedRecommender.load(path=ARTIFACTS_MODELS / "content_embed_model.joblib")
    RATE_FN["Content-Based (Embedding)"] = lambda u, i: cb_e.predict(urm.get(u, {}), i)
    RANK_FN["Content-Based (Embedding)"] = RATE_FN["Content-Based (Embedding)"]
except Exception as e:
    print("embedding content not loaded:", e)
try:
    lg = LightGCNRecommender.load(); RANK_FN["LightGCN"] = lambda u, i: lg.predict(u, i); lg_ok = True
except Exception as e:
    print("lightgcn not loaded:", e)
if genome_ok and lg_ok:
    try:
        dual = DualHeadHybrid.load()
        def dual_feats(u, i):
            return [cb_g.predict(urm.get(u, {}), i), uk.predict(u, i), ik.predict(u, i),
                    svd.predict(u, i), lg.predict(u, i), ip.get(i, 0), uc.get(u, 0), ip.get(i, 0)]
        RATE_FN["Dual-Head Hybrid"] = lambda u, i: dual.predict_rating_one(dual_feats(u, i))
        RANK_FN["Dual-Head Hybrid"] = lambda u, i: dual.rank_score_one(dual_feats(u, i))
    except Exception as e:
        print("dual-head not loaded:", e)

print(f"RATE_FN: {len(RATE_FN)} models | RANK_FN: {len(RANK_FN)} models")

RATE_FN: 10 models | RANK_FN: 12 models


## A. Comparison leaderboard (from `all_metrics.json`)

In [4]:
import json
from hybrid_recsys.config import ARTIFACTS_METRICS

metrics = json.loads((ARTIFACTS_METRICS / "all_metrics.json").read_text())
rows = []
for name, mm in metrics.items():
    row = {"Model": name, "RMSE": mm["rmse"], "MAE": mm["mae"]}
    for k in [5, 10, 20]:
        row[f"P@{k}"] = mm[f"k{k}"]["precision"]
        row[f"R@{k}"] = mm[f"k{k}"]["recall"]
        row[f"F1@{k}"] = mm[f"k{k}"]["f1"]
    rows.append(row)
board = pd.DataFrame(rows).set_index("Model").apply(pd.to_numeric, errors="coerce")
display(
    board.style
    .highlight_min(subset=["RMSE", "MAE"], color="#d4edda")
    .highlight_max(subset=["F1@5", "F1@10", "F1@20"], color="#d4edda")
    .format("{:.4f}", na_rep="-")
)

,RMSE,MAE,P@5,R@5,F1@5,P@10,R@10,F1@10,P@20,R@20,F1@20
Model,,,,,,,,,,,
Global Mean,1.0609,0.8339,0.0657,0.0430,0.0520,0.0661,0.0888,0.0758,0.0672,0.1868,0.0989
Popularity,2.7120,2.4856,0.6235,0.6380,0.6307,0.4791,0.8356,0.6090,0.3170,0.9464,0.4750
Content-Based,1.0462,0.7831,0.2409,0.1812,0.2069,0.1828,0.2394,0.2073,0.1278,0.3167,0.1821
User-Based k-NN,1.0401,0.8119,0.1082,0.0820,0.0933,0.1029,0.1500,0.1221,0.0896,0.2484,0.1317
Item-Based k-NN,0.8336,0.6223,0.4303,0.3786,0.4028,0.3537,0.5579,0.4329,0.2510,0.6877,0.3677
SVD,0.8108,0.6080,0.3673,0.3119,0.3373,0.2901,0.4482,0.3523,0.2118,0.5914,0.3119
Weighted Hybrid,0.8095,0.6074,0.3701,0.3145,0.3400,0.2917,0.4505,0.3541,0.2123,0.5921,0.3125
Stacked Hybrid,0.8054,0.6029,0.4384,0.3833,0.4090,0.3432,0.5334,0.4176,0.2389,0.6574,0.3504
Content-Based (Genome),0.9670,0.7091,0.4131,0.3623,0.3860,0.3174,0.4600,0.3756,0.2090,0.5269,0.2993


### Headline charts — RMSE/MAE and F1@10

In [5]:
dfp = board.reset_index()
rt = dfp.dropna(subset=["RMSE"])
fig = px.bar(rt.melt(id_vars="Model", value_vars=["RMSE", "MAE"]),
             x="Model", y="value", color="variable", barmode="group",
             title="RMSE and MAE by Model (rating-prediction models)",
             labels={"value": "Error", "variable": "Metric"})
fig.update_layout(xaxis_tickangle=-30)
save_fig(fig, "08_rmse_mae")

fig = px.bar(dfp.sort_values("F1@10", ascending=False), x="Model", y="F1@10",
             title="F1@10 by Model", color="F1@10", color_continuous_scale="Teal", text_auto=".3f")
fig.update_layout(coloraxis_showscale=False, xaxis_tickangle=-30)
save_fig(fig, "09_f1_at_10")

### Rating vs ranking trade-off (the key picture)

In [6]:
sc = dfp.dropna(subset=["RMSE"]).copy()
fig = px.scatter(sc, x="RMSE", y="F1@10", text="Model", color="Model",
                 title="Rating accuracy vs ranking quality (top-left = best)")
fig.update_traces(textposition="top center", marker_size=11)
fig.update_layout(showlegend=False, xaxis_title="RMSE (lower better)", yaxis_title="F1@10 (higher better)")
save_fig(fig, "16_rating_vs_ranking")

### F1 across K

In [7]:
long = []
for _, r in dfp.iterrows():
    for k in [5, 10, 20]:
        long.append({"Model": r["Model"], "K": k, "F1": r[f"F1@{k}"]})
fig = px.line(pd.DataFrame(long), x="K", y="F1", color="Model", markers=True,
              title="F1@K across models")
save_fig(fig, "17_f1_curves")

## B. NDCG@K & AUC (re-scored on a user sample)

F1@K is brittle to ties; NDCG (position-weighted) and AUC (ranking-pairs) are standard,
more robust ranking metrics. *Slow cell — re-ranks every model.*

In [8]:
from hybrid_recsys.evaluation.metrics import evaluate_ranking_extended

RANK_USERS = 400
ru = rng.choice(eval_user_ids, size=min(RANK_USERS, len(eval_user_ids)), replace=False)
rank_sample = test[test["userId"].isin(ru)]

rows = []
for name, fn in RANK_FN.items():
    res, auc = evaluate_ranking_extended(rank_sample, fn, train_val, all_movie_ids,
                                         n_negatives=N_NEGATIVES, k_values=[5, 10, 20],
                                         random_state=RANDOM_STATE)
    rows.append({"model": name, "NDCG@5": round(res[5]["ndcg"], 4), "NDCG@10": round(res[10]["ndcg"], 4),
                 "F1@10": round(res[10]["f1"], 4), "AUC": round(auc, 4)})
ndcg_df = pd.DataFrame(rows).set_index("model").sort_values("NDCG@10", ascending=False)
display(ndcg_df)

,NDCG@5,NDCG@10,F1@10,AUC
model,,,,
LightGCN,0.8043,0.8423,0.6218,0.9633
Popularity,0.7760,0.8167,0.6084,0.9766
Dual-Head Hybrid,0.5477,0.5637,0.4207,0.7683
Stacked Hybrid,0.5214,0.5458,0.4150,0.7613
Item-Based k-NN,0.4998,0.5441,0.4352,0.7385
Content-Based (Genome),0.5148,0.5167,0.3799,0.4986
Weighted Hybrid,0.4549,0.4679,0.3509,0.7426
SVD,0.4489,0.4664,0.3508,0.7423
Content-Based (Embedding),0.4743,0.4616,0.3300,0.3789


### Plot — NDCG@10 & AUC

In [9]:
b = ndcg_df.reset_index()
fig = px.bar(b.sort_values("NDCG@10", ascending=False).melt(id_vars="model", value_vars=["NDCG@10", "AUC"]),
             x="model", y="value", color="variable", barmode="group",
             title="NDCG@10 and AUC by model")
fig.update_layout(xaxis_tickangle=-30)
save_fig(fig, "18_ndcg_auc")

## C. Segmented RMSE — by user activity & item popularity

In [10]:
SEG_USERS = 500
seg = test[test["userId"].isin(rng.choice(eval_user_ids, size=min(SEG_USERS, len(eval_user_ids)), replace=False))].copy()
ucnt = train.groupby("userId").size(); ipop = train.groupby("movieId").size()
seg["user_bucket"] = pd.qcut(seg["userId"].map(ucnt), 4, labels=["Q1 least", "Q2", "Q3", "Q4 most"])
seg["item_bucket"] = pd.qcut(seg["movieId"].map(ipop).fillna(0).rank(method="first"),
                             4, labels=["cold", "Q2", "Q3", "popular"])
rows = []
for name, fn in RATE_FN.items():
    seg["_p"] = [fn(r.userId, r.movieId) for r in seg.itertuples()]
    for col, kind in [("user_bucket", "by user activity"), ("item_bucket", "by item popularity")]:
        for bk, g in seg.groupby(col, observed=True):
            e = (g["rating"].to_numpy() - g["_p"].to_numpy()); e = e[~np.isnan(e)]
            if len(e):
                rows.append({"model": name, "type": kind, "segment": str(bk),
                             "RMSE": round(float(np.sqrt((e ** 2).mean())), 4)})
seg_df = pd.DataFrame(rows)
display(seg_df.pivot_table(index=["type", "segment"], columns="model", values="RMSE"))

model                        Content-Based  Content-Based (Embedding)  \
type               segment                                              
by item popularity Q2               1.0736                     1.0715   
                   Q3               1.0509                     1.0432   
                   cold             1.0071                     1.0048   
                   popular          1.0410                     1.0387   
by user activity   Q1 least         1.0629                     1.0751   
                   Q2               1.0736                     1.0788   
                   Q3               1.0813                     1.1080   
                   Q4 most          0.9473                     0.8759   

model                        Content-Based (Genome)  Dual-Head Hybrid  \
type               segment                                              
by item popularity Q2                        0.9877            0.8092   
                   Q3                        0.9542            0.8008   
                   cold                      0.9516            0.7777   
                   popular                   0.9201            0.7821   
by user activity   Q1 least                  1.0139            0.8507   
                   Q2                        1.0066            0.8317   
                   Q3                        1.0171            0.8144   
                   Q4 most                   0.7419            0.6537   

model                        Global Mean  Item-Based k-NN     SVD  \
type               segment                                          
by item popularity Q2             1.0917           0.8372  0.8168   
                   Q3             1.0348           0.8208  0.8079   
                   cold           1.0196           0.7950  0.7876   
                   popular        1.0448           0.8061  0.7859   
by user activity   Q1 least       1.0941           0.8606  0.8585   
                   Q2             1.0837           0.8605  0.8396   
                   Q3             1.0589           0.8423  0.8193   
                   Q4 most        0.9456           0.6775  0.6619   

model                        Stacked Hybrid  User-Based k-NN  Weighted Hybrid  
type               segment                                                     
by item popularity Q2                0.8111           1.0766           0.8151  
                   Q3                0.8017           1.0070           0.8077  
                   cold              0.7754           0.9934           0.7852  
                   popular           0.7827           1.0216           0.7866  
by user activity   Q1 least          0.8503           1.0625           0.8579  
                   Q2                0.8337           1.0539           0.8373  
                   Q3                0.8136           1.0317           0.8191  
                   Q4 most           0.6541           0.9457           0.6613

### Plot — RMSE by user-activity bucket

In [11]:
focus_models = [m for m in ["Content-Based", "Content-Based (Genome)", "SVD", "Stacked Hybrid", "Dual-Head Hybrid"] if m in RATE_FN]
f = seg_df[(seg_df["type"] == "by user activity") & seg_df["model"].isin(focus_models)]
fig = px.line(f, x="segment", y="RMSE", color="model", markers=True,
              title="RMSE by user-activity bucket (lower = better)")
save_fig(fig, "19_segmented_user")

## D. Beyond-accuracy — coverage, diversity, novelty

In [12]:
from hybrid_recsys.evaluation.metrics import catalogue_coverage, intra_list_diversity, novelty
from hybrid_recsys.pipeline.features import load_item_features

itf, pos_of = load_item_features()
dense = itf.toarray().astype("float32"); nrm = np.linalg.norm(dense, axis=1, keepdims=True); nrm[nrm == 0] = 1.0
dn = dense / nrm
def feat(m):
    p = pos_of.get(m); return dn[p] if p is not None else None

DIV_USERS, CAND = 60, 1000
duser = rng.choice(eval_user_ids, size=min(DIV_USERS, len(eval_user_ids)), replace=False)
candpool = rng.choice(all_movie_ids, size=CAND, replace=False)
n_inter = len(train)
rows = []
for name, fn in RANK_FN.items():
    lists = [list(top_n(fn, u, set(urm.get(u, {})), candpool, movies, n=10)["movieId"]) for u in duser]
    rows.append({"model": name,
                 "coverage": round(catalogue_coverage(lists, len(movies)), 4),
                 "diversity": round(intra_list_diversity(lists, feat), 4),
                 "novelty": round(novelty(lists, ip, n_inter), 3)})
bey = pd.DataFrame(rows).set_index("model")
display(bey)

,coverage,diversity,novelty
model,,,
Global Mean,0.0002,0.8157,23.270
Content-Based,0.0033,0.7287,18.523
User-Based k-NN,0.0007,0.8094,22.122
Item-Based k-NN,0.0017,0.6894,14.543
SVD,0.0021,0.7347,16.668
Weighted Hybrid,0.0021,0.7330,16.636
Stacked Hybrid,0.0017,0.6860,14.728
Popularity,0.0004,0.7665,10.033
Content-Based (Genome),0.0028,0.7147,14.854


### Plot — diversity vs novelty (bubble = coverage)

In [13]:
b = bey.reset_index()
fig = px.scatter(b, x="novelty", y="diversity", size="coverage", color="model", text="model",
                 title="Beyond-accuracy: diversity vs novelty (bubble = coverage)")
fig.update_traces(textposition="top center")
save_fig(fig, "20_beyond_accuracy")

## E. Bootstrap confidence intervals on RMSE

In [14]:
from hybrid_recsys.evaluation.report import bootstrap_ci

cand_models = [m for m in ["SVD", "Weighted Hybrid", "Stacked Hybrid", "Dual-Head Hybrid",
                           "Item-Based k-NN", "Content-Based (Genome)"] if m in RATE_FN]
boot = []
for name in cand_models:
    fn = RATE_FN[name]
    p = np.array([fn(r.userId, r.movieId) for r in test_sample.itertuples()])
    se = (test_sample["rating"].to_numpy() - p) ** 2
    pt, lo, hi = bootstrap_ci(se, n_boot=500, agg="rmse", random_state=RANDOM_STATE)
    boot.append({"model": name, "RMSE": pt, "CI_low": lo, "CI_high": hi})
bdf = pd.DataFrame(boot)
display(bdf)
fig = px.scatter(bdf, x="model", y="RMSE",
                 error_y=bdf["CI_high"] - bdf["RMSE"], error_y_minus=bdf["RMSE"] - bdf["CI_low"],
                 title="RMSE with 95% bootstrap CI (test-sample users)")
save_fig(fig, "21_bootstrap_rmse")

,model,RMSE,CI_low,CI_high
0,SVD,0.8013,0.7897,0.8127
1,Weighted Hybrid,0.8006,0.7890,0.8121
2,Stacked Hybrid,0.7959,0.7836,0.8076
3,Dual-Head Hybrid,0.7945,0.7825,0.8063
4,Item-Based k-NN,0.8176,0.8043,0.8295
5,Content-Based (Genome),0.9608,0.9460,0.9771


## F. Cold-start simulation — content models with only 3 ratings

For a sample of users we keep **only their first 3 training ratings** as history and
re-evaluate the content models (which depend only on history). Shows graceful behaviour
when a user is almost new.

In [15]:
cold_models = [m for m in ["Content-Based", "Content-Based (Genome)", "Content-Based (Embedding)"] if m in RATE_FN]
cmap = {"Content-Based": cb}
if "Content-Based (Genome)" in RATE_FN: cmap["Content-Based (Genome)"] = cb_g
if "Content-Based (Embedding)" in RATE_FN: cmap["Content-Based (Embedding)"] = cb_e

cu = rng.choice(eval_user_ids, size=min(400, len(eval_user_ids)), replace=False)
cold_hist = {u: dict(list(urm.get(u, {}).items())[:3]) for u in cu}
cseg = test[test["userId"].isin(cu)]
rows = []
for name in cold_models:
    model = cmap[name]
    full = np.array([model.predict(urm.get(r.userId, {}), r.movieId) for r in cseg.itertuples()])
    cold = np.array([model.predict(cold_hist.get(r.userId, {}), r.movieId) for r in cseg.itertuples()])
    def _rmse(p):
        e = cseg["rating"].to_numpy() - p; e = e[~np.isnan(e)]; return round(float(np.sqrt((e ** 2).mean())), 4)
    rows.append({"model": name, "RMSE (full history)": _rmse(full), "RMSE (3 ratings)": _rmse(cold)})
display(pd.DataFrame(rows))

,model,RMSE (full history),RMSE (3 ratings)
0,Content-Based,1.0661,1.2708
1,Content-Based (Genome),0.9634,1.2712
2,Content-Based (Embedding),1.0583,1.2735


## G. Full-catalogue ranking sanity (no sampling)

Sampled-100-negative metrics can disagree with exact ones (Krichene & Rendle, 2020). Here a
**full-catalogue** ranking pass on a small user sample for a few fast models, to check the
sampled F1@10 ordering holds. *Slow cell.*

In [16]:
from hybrid_recsys.evaluation.metrics import evaluate_ranking

FULL_USERS = 80
fu = rng.choice(eval_user_ids, size=min(FULL_USERS, len(eval_user_ids)), replace=False)
full_sample = test[test["userId"].isin(fu)]
check = [m for m in ["SVD", "Item-Based k-NN", "Stacked Hybrid", "LightGCN", "Dual-Head Hybrid"] if m in RANK_FN]
rows = []
for name in check:
    res = evaluate_ranking(full_sample, RANK_FN[name], train_val, k_values=[10], random_state=RANDOM_STATE)
    rows.append({"model": name, "F1@10 (full-catalogue)": round(res[10]["f1"], 4)})
display(pd.DataFrame(rows))

,model,F1@10 (full-catalogue)
0,SVD,0.0477
1,Item-Based k-NN,0.0437
2,Stacked Hybrid,0.0534
3,LightGCN,0.0257
4,Dual-Head Hybrid,0.0659


## Conclusion

- **A** gives the headline leaderboard + the rating-vs-ranking trade-off picture (the
  Dual-Head should sit toward the top-left).
- **B** (NDCG/AUC) is the more robust ranking verdict; **C** shows *where* models win
  (cold/sparse segments); **D** the accuracy-vs-diversity trade-off; **E** whether RMSE
  gaps are statistically real; **F** cold-start behaviour; **G** that sampled ranking
  isn't an artifact of the 100-negative protocol.
- Report rating models (RMSE/MAE) and ranking models (NDCG/AUC/F1) in separate tables;
  note Popularity's high F1 is a sampled-negatives popularity bias.